# 🎙️ Chatterbox TTS - Voice Cloning

Clone any voice from a short audio sample and generate speech in that voice — free, open-source, runs entirely in this notebook.

### ⚠️ Before you start
1. Go to **Runtime → Change runtime type** and select **T4 GPU**
2. Then use **Runtime → Run all** to run every cell in order automatically

### Limitations
- **English only** — this notebook loads the base Chatterbox model, which is English-only. (A multilingual version supporting 23+ languages exists, but isn't set up in this notebook.)
- Free Colab sessions disconnect after ~90 min of inactivity or ~12 hours max — just rerun the cells if that happens
- Best voice samples: 10-30 seconds, clean audio, minimal background noise

---

## Step 1: Install Dependencies
Run this first. It installs everything needed and **restarts the kernel automatically** — this is expected, just wait for it to finish before moving to Step 2.


In [ ]:
import subprocess
import sys

def run_command(command, description=""):
    print(f"Running: {description if description else command}")
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Warning: {description} failed")
        print(f"stderr: {result.stderr}")
    else:
        print(f"Success: {description}")
    return result.returncode == 0

print(f"Python version: {sys.version}")

run_command("pip install --upgrade pip", "Upgrading pip")

print("🧹 Cleaning existing installations...")
run_command("pip uninstall -y torch torchvision torchaudio transformers chatterbox-tts accelerate huggingface-hub diffusers torchao perth resemble-perth tensorflow tensorflow-cpu tf-keras", "Removing conflicting packages")

print("📦 Installing PyTorch 2.5.0...")
run_command("pip install torch==2.5.0 torchaudio==2.5.0", "Installing PyTorch 2.5.0")

print("📦 Installing Chatterbox dependencies...")
run_command("pip install transformers==4.46.3", "Installing transformers 4.46.3")
run_command("pip install diffusers==0.29.0", "Installing diffusers 0.29.0")
run_command("pip install 'huggingface_hub>=0.23.0,<0.25.0'", "Installing huggingface_hub")
run_command("pip install 'accelerate>=0.25.0'", "Installing accelerate")

run_command("apt update && apt install -y git-lfs", "Installing git-lfs")

print("📦 Installing audio processing libraries...")
run_command("pip install 'numpy>=1.24.0,<1.26.0' librosa==0.11.0 safetensors soundfile scipy", "Installing audio libs")

print("📦 Installing resemble-perth...")
run_command("pip install resemble-perth", "Installing resemble-perth")

print("📦 Installing s3tokenizer and conformer...")
run_command("pip install s3tokenizer conformer", "Installing s3tokenizer and conformer")

print("📦 Installing Chatterbox TTS...")
run_command("pip install chatterbox-tts --no-deps", "Installing Chatterbox TTS without dependencies")

print("🔧 Fixing protobuf...")
run_command("pip uninstall -y protobuf", "Uninstalling protobuf")
run_command("pip install protobuf==3.20.3", "Installing protobuf 3.20.3")

print("\n✅ Installation complete!")
print("🔄 Restarting kernel...")
print("⚠️  WAIT for kernel restart, then run Cell 2!")
get_ipython().kernel.do_shutdown(True)

## Step 2: Load Model & Connect Google Drive
This verifies the install worked, mounts your Google Drive (to save generated audio), and loads the Chatterbox model onto the GPU.


In [ ]:
import sys
print("🔍 Verifying installation...\n")

try:
    import torch
    import torchaudio
    print(f"✅ PyTorch {torch.__version__} imported successfully")
    print(f"   CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"   CUDA version: {torch.version.cuda}")
except ImportError as e:
    print(f"❌ PyTorch import error: {e}")
    sys.exit(1)

try:
    import transformers
    print(f"✅ Transformers {transformers.__version__} imported successfully")
except ImportError as e:
    print(f"❌ Transformers import error: {e}")
    sys.exit(1)

try:
    from chatterbox.tts import ChatterboxTTS
    print("✅ ChatterboxTTS imported successfully")
except Exception as e:
    print(f"❌ ChatterboxTTS import error: {e}")
    sys.exit(1)

from google.colab import drive
import os

def setup_drive():
    try:
        drive.mount('/content/drive')
        drive_path = '/content/drive/MyDrive/Chatterbox'
        os.makedirs(drive_path, exist_ok=True)
        print(f"✅ Drive setup complete: {drive_path}")
        return drive_path
    except Exception as e:
        print(f"❌ Drive setup failed: {e}")
        return None

DRIVE_PATH = setup_drive()

def load_model():
    try:
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print(f"Loading model on device: {device}")
        model = ChatterboxTTS.from_pretrained(device=device)
        print("✅ Model loaded successfully")
        return model
    except Exception as e:
        print(f"❌ Model loading failed: {e}")
        print("Trying CPU fallback...")
        model = ChatterboxTTS.from_pretrained(device="cpu")
        print("✅ Model loaded successfully on CPU")
        return model

model = load_model()
print("\n🎉 Ready! Proceed to Cell 3.")

## Step 3: Launch the Voice Cloning App
This starts a Gradio web interface. Once it runs, click the `*.gradio.live` link in the output — that's where you upload your voice sample, type text, and generate speech.


In [ ]:
!pip install gradio --no-deps -q
!pip install "huggingface_hub>=0.23.0,<0.25.0" --force-reinstall --no-deps -q

import gradio as gr
import torch
import torchaudio
import os

# --- Ensure Google Drive is mounted ---
if not os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/Chatterbox'
os.makedirs(DRIVE_PATH, exist_ok=True)

# --- Ensure model is loaded, regardless of what ran before ---
if 'model' not in globals():
    print("⚠️ Model not found in memory — loading it now...")
    from chatterbox.tts import ChatterboxTTS
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Loading model on device: {device}")
    try:
        model = ChatterboxTTS.from_pretrained(device=device)
    except Exception as e:
        print(f"❌ Failed on {device}, trying CPU fallback: {e}")
        model = ChatterboxTTS.from_pretrained(device="cpu")
    print("✅ Model loaded successfully")
else:
    print("✅ Model already loaded in memory, reusing it.")

def split_into_chunks(text, max_words=40):
    sentences = text.strip().replace('\n', ' ').split('.')
    sentences = [s.strip() for s in sentences if s.strip()]
    chunks, current_chunk, current_word_count = [], "", 0
    for sentence in sentences:
        sentence_words = sentence.split()
        if current_word_count + len(sentence_words) > max_words and current_chunk:
            chunks.append(current_chunk.strip() + ".")
            current_chunk = sentence
            current_word_count = len(sentence_words)
        else:
            current_chunk = current_chunk + ". " + sentence if current_chunk else sentence
            current_word_count += len(sentence_words)
    if current_chunk:
        chunks.append(current_chunk.strip() + ".")
    return chunks

def generate_speech_ui(text, voice_file, exaggeration, cfg_weight, max_chunk_words):
    global model
    if 'model' not in globals() or model is None:
        return None, "❌ Model is not loaded. Restart the runtime and rerun this cell."

    if not text.strip():
        return None, "❌ Please enter some text."

    voice_path = voice_file if voice_file else None
    chunks = split_into_chunks(text, int(max_chunk_words))
    wav_tensors = []

    for chunk in chunks:
        gen_params = {
            "text": chunk,
            "exaggeration": exaggeration,
            "cfg_weight": cfg_weight
        }
        if voice_path:
            gen_params["audio_prompt_path"] = voice_path

        try:
            wav = model.generate(**gen_params)
            wav_tensors.append(wav)
        except Exception as e:
            return None, f"❌ Generation failed on chunk: {e}"

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if not wav_tensors:
        return None, "❌ No audio generated."

    full_audio = torch.cat(wav_tensors, dim=1)
    output_path = f"{DRIVE_PATH}/generated_speech.wav"
    torchaudio.save(output_path, full_audio, model.sr)

    return output_path, f"✅ Generated {len(chunks)} chunk(s), saved to {output_path}"

with gr.Blocks(title="Chatterbox Voice Cloner") as demo:
    gr.Markdown("## 🎙️ Chatterbox Voice Cloning")
    gr.Markdown("Upload a clean 10-30s voice sample (WAV), type your text, and generate.")

    with gr.Row():
        with gr.Column():
            text_input = gr.Textbox(label="Text to speak", lines=5, placeholder="Type what you want the voice to say...")
            voice_input = gr.Audio(label="Voice sample to clone (optional)", type="filepath")
            exaggeration = gr.Slider(0.0, 1.0, value=0.75, label="Exaggeration")
            cfg_weight = gr.Slider(0.0, 1.0, value=0.3, label="CFG Weight")
            max_chunk_words = gr.Slider(10, 100, value=40, step=5, label="Max words per chunk")
            generate_btn = gr.Button("🎙️ Generate Speech", variant="primary")
        with gr.Column():
            audio_output = gr.Audio(label="Generated Audio")
            status_output = gr.Textbox(label="Status")

    generate_btn.click(
        fn=generate_speech_ui,
        inputs=[text_input, voice_input, exaggeration, cfg_weight, max_chunk_words],
        outputs=[audio_output, status_output]
    )

demo.launch(share=True, debug=True)